In [1]:
import os
import csv
import itertools
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt

# --------- Configuration ---------
DATA_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\data\split_with_713"
OUTPUT_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\outputs\plot\final\cnn9_trans_02"
NUM_CLASSES = 39
BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 30
WEIGHT_DECAY = 1e-4
LR = 1e-3
PATIENCE = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.backends.cudnn.benchmark = True

# --------- Transforms ---------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --------- Dataloaders ---------

def get_dataloaders(data_dir, batch_size, num_workers):
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_ds = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
    test_ds = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
    )

# --------- CNN + Transformer Model ---------
class CNNTransformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.embedding_dim = 128
        self.seq_len = 56 * 56
        self.linear_proj = nn.Linear(self.embedding_dim, 128)
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)
        x = self.linear_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

model = CNNTransformer(NUM_CLASSES).to(DEVICE)

# --------- Optimizer & Loss ---------
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

# --------- Evaluation ---------

def evaluate_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().tolist()
            y_pred.extend(preds)
            y_true.extend(labels.tolist())
    return y_true, y_pred

# --------- Training Loop ---------

def train():
    print(">> start:", flush=True)
    train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS)

    best_val_acc = 0.0
    epochs_no_improve = 0

    epochs, train_losses, val_accs, val_f1s = [], [], [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)

        y_true_val, y_pred_val = evaluate_predictions(model, val_loader)
        val_acc = accuracy_score(y_true_val, y_pred_val)
        val_f1 = f1_score(y_true_val, y_pred_val, average='macro')

        epochs.append(epoch)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        val_f1s.append(val_f1)

        print(f"Epoch {epoch}/{NUM_EPOCHS} - Loss: {train_loss:.4f} - Acc: {val_acc:.4f} - F1: {val_f1:.4f}", flush=True)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_cnn_with_trans.pth"))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    # --------- Final Test Evaluation ---------
    y_true_test, y_pred_test = evaluate_predictions(model, test_loader)
    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_prec = precision_score(y_true_test, y_pred_test, average='macro')
    test_rec = recall_score(y_true_test, y_pred_test, average='macro')
    test_f1 = f1_score(y_true_test, y_pred_test, average='macro')

    print("\n===== FINAL TEST METRICS =====")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall   : {test_rec:.4f}")
    print(f"F1-Score : {test_f1:.4f}\n")

    # --------- Confusion Matrix ---------
    cm = confusion_matrix(y_true_test, y_pred_test)
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix — Test Set')
    plt.colorbar(shrink=0.8)
    ticks = range(NUM_CLASSES)
    plt.xticks(ticks)
    plt.yticks(ticks)
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'), ha='center', va='center', color='white' if cm[i, j] > thresh else 'black')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_test.png'), dpi=300)
    plt.close()

    # --------- Save Metrics CSV ---------
    with open(os.path.join(OUTPUT_DIR, 'metrics.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_acc', 'val_f1'])
        for e, l, a, f1_value in zip(epochs, train_losses, val_accs, val_f1s):
            writer.writerow([e, l, a, f1_value])

    # --------- Plot Curves ---------
    def plot_curve(values, ylabel, filename):
        plt.figure()
        plt.plot(epochs, values, marker='o')
        plt.title(f'{ylabel} vs. Epoch')
        plt.xlabel('Epoch')
        plt.ylabel(ylabel)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
        plt.close()

    plot_curve(train_losses, 'Loss', 'training_loss.png')
    plot_curve(val_accs, 'Accuracy', 'validation_accuracy.png')
    plot_curve(val_f1s, 'F1-Score', 'validation_f1.png')


if __name__ == '__main__':
    train()

>> start:
Epoch 1/30 - Loss: 2.8936 - Acc: 0.3650 - F1: 0.2057
Epoch 2/30 - Loss: 2.1422 - Acc: 0.5696 - F1: 0.4461
Epoch 3/30 - Loss: 1.4990 - Acc: 0.6803 - F1: 0.6253
Epoch 4/30 - Loss: 1.1312 - Acc: 0.8181 - F1: 0.7847
Epoch 5/30 - Loss: 0.9203 - Acc: 0.8228 - F1: 0.7907
Epoch 6/30 - Loss: 0.7965 - Acc: 0.8948 - F1: 0.8743
Epoch 7/30 - Loss: 0.7080 - Acc: 0.8738 - F1: 0.8517
Epoch 8/30 - Loss: 0.6276 - Acc: 0.9010 - F1: 0.8790
Epoch 9/30 - Loss: 0.5922 - Acc: 0.9112 - F1: 0.8964
Epoch 10/30 - Loss: 0.5421 - Acc: 0.9062 - F1: 0.8906
Epoch 11/30 - Loss: 0.4953 - Acc: 0.9098 - F1: 0.8932
Epoch 12/30 - Loss: 0.4578 - Acc: 0.9294 - F1: 0.9179
Epoch 13/30 - Loss: 0.4266 - Acc: 0.9210 - F1: 0.9094
Epoch 14/30 - Loss: 0.4056 - Acc: 0.9382 - F1: 0.9291
Epoch 15/30 - Loss: 0.3843 - Acc: 0.9522 - F1: 0.9432
Epoch 16/30 - Loss: 0.3489 - Acc: 0.9520 - F1: 0.9427
Epoch 17/30 - Loss: 0.3332 - Acc: 0.9602 - F1: 0.9539
Epoch 18/30 - Loss: 0.3124 - Acc: 0.9599 - F1: 0.9544
Epoch 19/30 - Loss: 0.2926 